In [ ]:
import pandas as pd
df = pd.read_csv('/Users/mmejiaj/Documents/Unibo_python/cil-intrusion-detection/data/raw/UNSW_NB15_training-set.csv')
df_test = pd.read_csv('/Users/mmejiaj/Documents/Unibo_python/cil-intrusion-detection/data/raw/UNSW_NB15_testing-set.csv')


In [ ]:
from sklearn.preprocessing import OneHotEncoder
import pandas as pd

cat_vars = ['proto', 'service', 'state']

df = df.drop_duplicates()
df = df.drop(columns=['id', 'label', 'is_sm_ips_ports'], errors='ignore')
df_test = df_test.drop_duplicates()
df_test = df_test.drop(columns=['id', 'label', 'is_sm_ips_ports'], errors='ignore')

encoder = OneHotEncoder(
    sparse_output=False,      # devuelve array denso (mejor para pandas/NN)
    handle_unknown='ignore'   # evita crash si aparece categoría nueva
)

encoded_array = encoder.fit_transform(df[cat_vars])
encoded_array_test = encoder.transform(df_test[cat_vars])

encoded_cols = encoder.get_feature_names_out(cat_vars)

df_encoded = pd.DataFrame(encoded_array, columns=encoded_cols, index=df.index)
df_encoded_test = pd.DataFrame(encoded_array_test, columns=encoded_cols, index=df_test.index)


df = pd.concat([df.drop(columns=cat_vars), df_encoded], axis=1)
df = df[df['attack_cat'] != 'Worms']

df_test = pd.concat([df_test.drop(columns=cat_vars), df_encoded_test], axis=1)
df_test = df_test[df_test['attack_cat'] != 'Worms']

In [ ]:
import zipfile
from pathlib import Path
import tempfile
import shutil

def export_zip_with_root(df, df_test, zip_path):
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)

    temp_dir = Path(tempfile.mkdtemp())

    # 🔴 ROOT "2015" (clave para tu loader)
    root = temp_dir / "2015"
    train_dir = root / "train"
    test_dir = root / "test"

    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    # TRAIN CSV per class
    for attack in sorted(df['attack_cat'].unique()):
        subset = df[df['attack_cat'] == attack]
        safe_name = str(attack).strip().replace(" ", "_").replace("/", "_")
        subset.to_csv(train_dir / f"{safe_name}.csv", index=False)

    # TEST CSV per class
    for attack in sorted(df_test['attack_cat'].unique()):
        subset = df_test[df_test['attack_cat'] == attack]
        safe_name = str(attack).strip().replace(" ", "_").replace("/", "_")
        subset.to_csv(test_dir / f"{safe_name}.csv", index=False)

    # Crear ZIP preservando la raíz 2015/
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
        for file in temp_dir.rglob("*.csv"):
            arcname = file.relative_to(temp_dir)  # mantiene "2015/train/..."
            z.write(file, arcname)

    shutil.rmtree(temp_dir)
    print(f"ZIP compatible con loader creado en: {zip_path}")

In [ ]:
export_zip_with_root(df, df_test, "prueba.zip")